# Commodity silver-equivalence index

Use authorized, cited observations and explicit silver-per-unit conversion ranges. No historical rate, commodity unit, inflation adjustment, or period order is assumed. The selected baseline is 100 within each region/commodity/unit series; reported ranges preserve the supplied input uncertainty.

In [ ]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
from uuid import UUID

from socio_economic.silver_equivalence import (
    calculate_silver_equivalence_index,
    read_observations_csv,
    sha256_file,
    write_index_svg,
)

input_csv = Path(os.environ.get("PLEROMA_ECONOMIC_INPUT", ""))
output_dir = Path(os.environ.get("PLEROMA_OUTPUT_DIR", ""))
baseline_period = os.environ.get("PLEROMA_BASELINE_PERIOD", "")
parameter_set_id = UUID(os.environ["PLEROMA_PARAMETER_SET_ID"])
notebook_revision = os.environ["PLEROMA_NOTEBOOK_REVISION"]
environment_reference = os.environ["PLEROMA_ENVIRONMENT_REFERENCE"]
commodity_uri = os.environ["PLEROMA_COMMODITY_URI"]
region_id = UUID(os.environ["PLEROMA_REGION_ID"])
commodity_unit = os.environ["PLEROMA_COMMODITY_UNIT"]

if not input_csv.is_file() or not output_dir.is_dir() or not baseline_period:
    raise ValueError("Set the authorized input, existing output directory, and baseline period")

In [ ]:
observations = read_observations_csv(input_csv)
report = calculate_silver_equivalence_index(
    observations,
    baseline_period=baseline_period,
)
payload = report.writeback_payload(
    input_sha256=sha256_file(input_csv),
    parameter_set_id=parameter_set_id,
    notebook_revision=notebook_revision,
    environment_reference=environment_reference,
    executed_at=datetime.now(timezone.utc),
)
chart_path = output_dir / "economic_index.svg"
payload_path = output_dir / "economic_index_payload.json"
if payload_path.exists():
    raise FileExistsError(f"Refusing to overwrite {payload_path}")
write_index_svg(
    report,
    chart_path,
    commodity_uri=commodity_uri,
    region_id=region_id,
    commodity_unit=commodity_unit,
)
payload_path.write_text(json.dumps(payload, allow_nan=False, indent=2) + "\n", encoding="utf-8")
payload

The series and payload are not written to a database here. Validate the output references and submit the structured payload through the authorized writeback path; register the SVG as a computational-output Media_Asset.